# 额外的周末练习 - 第 2 周

现在，使用您从第 2 周学到的所有知识为您在第 1 周练习中构建的技术问题/回答器构建完整的原型。

这应该包括 Gradio UI、流媒体、使用系统提示来添加专业知识以及在模型之间切换的能力。如果您能够演示工具的使用，则可获得奖励积分！

如果您觉得大胆，请看看是否可以添加音频输入，以便您可以与它交谈，并让它用音频进行响应。 ChatGPT 或 Claude 可以帮助您，如果您有疑问，也可以给我发电子邮件。

我很快就会在这里发布完整的解决方案 - 除非有人比我先一步......

这方面的商业应用有很多，从语言导师到公司入职解决方案，再到人工智能伴侣和课程（就像这个！），我迫不及待地想看到你的结果。

In [ ]:
# 进口
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr 
from IPython.display import Markdown, display

In [ ]:
# 加载环境变量并检查 API 密钥

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set (optional)")

In [ ]:
# 初始化模型客户端

openai = OpenAI()

# 设置替代模型端点
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url) if google_api_key else None
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# 模型映射
MODELS = {
    "GPT-4.1-mini": {"client": openai, "model": "gpt-4.1-mini"},
    "Claude Sonnet 4.5": {"client": anthropic, "model": "claude-sonnet-4-5-20250929"},
    "Gemini 2.5 Flash": {"client": gemini, "model": "gemini-2.5-flash-lite"},
    "Llama 3.2 (Local)": {"client": ollama, "model": "llama3.2"}
}

print("Model clients initialized successfully!")

In [ ]:
# SQL专家的系统提示

system_prompt = """You are an expert SQL database engineer with deep knowledge of SQL syntax, optimization, and best practices.

Your role is to:
1. Generate accurate, efficient SQL queries from natural language questions
2. Use only the tables and columns provided in the database schema
3. Follow standard SQL syntax (PostgreSQL/SQLite compatible)
4. Provide clear explanations of your queries
5. Suggest optimizations when relevant

When responding:
- Return the SQL query in a fenced code block (```sql)
- Add a brief explanation of what the query does
- If the question is ambiguous, make reasonable assumptions and state them
- If a schema is not provided, generate generic SQL with common table/column names

Always prioritize correctness and clarity."""

In [ ]:
# 流式SQL生成功能

def generate_sql_stream(question, schema, model_name):
    """
    Generate SQL query from natural language with streaming response
    """
    model_info = MODELS.get(model_name)
    if not model_info or model_info["client"] is None:
        yield "Error: Selected model is not available. Please check your API keys."
        return
    
    client = model_info["client"]
    model = model_info["model"]
    
    user_content = f"Database Schema:\n{schema.strip()}\n\nQuestion: {question}" if schema.strip() else f"Question: {question}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        
        result = ""
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            result += delta
            yield result
            
    except Exception as e:
        yield f"Error: {str(e)}"

In [ ]:
# 奖励：用于执行 SQL 查询的工具

DB = "sql_demo.db"

def execute_sql_query(query):
    """
    Execute a SQL query against a demo database
    Returns the results as a formatted string
    """
    print(f"TOOL CALLED: Executing SQL query", flush=True)
    try:
        with sqlite3.connect(DB) as conn:
            cursor = conn.cursor()
            cursor.execute(query)
            results = cursor.fetchall()
            
            if not results:
                return "Query executed successfully. No results returned."
            
            column_names = [description[0] for description in cursor.description]
            result_str = f"Results ({len(results)} rows):\n\n"
            result_str += " | ".join(column_names) + "\n"
            result_str += "-" * (len(result_str) - 1) + "\n"
            
            for row in results:
                result_str += " | ".join(str(val) for val in row) + "\n"
            
            return result_str
    except Exception as e:
        return f"Error executing query: {str(e)}"

# 使用示例数据初始化演示数据库
def init_demo_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        
        cursor.execute('DROP TABLE IF EXISTS orders')
        cursor.execute('DROP TABLE IF EXISTS customers')
        cursor.execute('DROP TABLE IF EXISTS products')
        
        cursor.execute('''
            CREATE TABLE customers (
                id INTEGER PRIMARY KEY,
                name TEXT,
                email TEXT,
                created_at DATE
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE orders (
                id INTEGER PRIMARY KEY,
                customer_id INTEGER,
                total REAL,
                order_date DATE,
                FOREIGN KEY (customer_id) REFERENCES customers(id)
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE products (
                id INTEGER PRIMARY KEY,
                name TEXT,
                price REAL
            )
        ''')
        
        cursor.executemany('INSERT INTO customers VALUES (?, ?, ?, ?)', [
            (1, 'Alice Johnson', 'alice@email.com', '2025-01-15'),
            (2, 'Bob Smith', 'bob@email.com', '2025-12-20'),
            (3, 'Carol White', 'carol@email.com', '2026-02-01'),
            (4, 'David Brown', 'david@email.com', '2026-02-15')
        ])
        
        cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?)', [
            (1, 1, 150.00, '2026-02-01'),
            (2, 1, 200.00, '2026-02-15'),
            (3, 2, 75.00, '2025-12-25'),
            (4, 3, 300.00, '2026-02-20'),
            (5, 4, 450.00, '2026-02-28')
        ])
        
        cursor.executemany('INSERT INTO products VALUES (?, ?, ?)', [
            (1, 'Laptop', 999.99),
            (2, 'Mouse', 29.99),
            (3, 'Keyboard', 79.99),
            (4, 'Monitor', 299.99)
        ])
        
        conn.commit()
    print("Demo database initialized with sample data!")

init_demo_db()

In [ ]:
# SQL执行的工具定义

execute_sql_function = {
    "name": "execute_sql_query",
    "description": "Execute a SQL query against the demo database and return the results. Use this when the user wants to run or test a generated SQL query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The SQL query to execute"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": execute_sql_function}]

In [ ]:
# 增强的聊天功能，支持工具调用

def handle_tool_calls(message):
    """Handle tool calls from the LLM"""
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "execute_sql_query":
            arguments = json.loads(tool_call.function.arguments)
            query = arguments.get('query')
            result = execute_sql_query(query)
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    return responses

def chat_with_tools(question, schema, model_name, enable_tools):
    """
    Chat function that supports tool calling for SQL execution
    """
    model_info = MODELS.get(model_name)
    if not model_info or model_info["client"] is None:
        yield "Error: Selected model is not available. Please check your API keys."
        return
    
    client = model_info["client"]
    model = model_info["model"]
    
    user_content = f"Database Schema:\n{schema.strip()}\n\nQuestion: {question}" if schema.strip() else f"Question: {question}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    
    try:
        if enable_tools:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                tools=tools,
                stream=False
            )
            
            while response.choices[0].finish_reason == "tool_calls":
                message = response.choices[0].message
                tool_responses = handle_tool_calls(message)
                messages.append(message)
                messages.extend(tool_responses)
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=tools,
                    stream=False
                )
            
            yield response.choices[0].message.content
        else:
            stream = client.chat.completions.create(
                model=model,
                messages=messages,
                stream=True
            )
            
            result = ""
            for chunk in stream:
                delta = chunk.choices[0].delta.content or ""
                result += delta
                yield result
            
    except Exception as e:
        yield f"Error: {str(e)}"

In [ ]:
# 奖励：音频输入/输出功能

def transcribe_audio(audio_file):
    """Convert audio input to text using Whisper"""
    if audio_file is None:
        return ""
    
    try:
        with open(audio_file, "rb") as f:
            transcript = openai.audio.transcriptions.create(
                model="whisper-1",
                file=f
            )
        return transcript.text
    except Exception as e:
        print(f"Audio transcription error: {str(e)}")
        return ""

def text_to_speech(text):
    """Convert text response to speech"""
    try:
        response = openai.audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="alloy",
            input=text
        )
        return response.content
    except Exception as e:
        print(f"Text-to-speech error: {str(e)}")
        return None

In [ ]:
# Gradio UI - 带音频支持的全功能 SQL 生成器

def process_query(audio_input, text_input, schema, model_name, enable_tools, enable_audio_output):
    """
    Main processing function that handles both audio and text input
    """
    question = text_input
    
    if audio_input is not None:
        transcribed = transcribe_audio(audio_input)
        if transcribed:
            question = transcribed
    
    if not question.strip():
        yield "Please provide a question either via text or audio.", None
        return
    
    result = ""
    for chunk in chat_with_tools(question, schema, model_name, enable_tools):
        result = chunk
        yield chunk, None
    
    if enable_audio_output:
        audio = text_to_speech(result)
        yield result, audio
    else:
        yield result, None

# 默认模式
default_schema = """Table: customers (id, name, email, created_at)
Table: orders (id, customer_id, total, order_date)
Table: products (id, name, price)"""

# 构建用户界面
with gr.Blocks(title="SQL Query Generator") as demo:
    gr.Markdown("# 🗄️ Natural Language to SQL Query Generator")
    gr.Markdown("Ask questions in natural language and get SQL queries generated by AI. Optionally use your voice!")
    
    with gr.Row():
        with gr.Column(scale=2):
            audio_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="🎤 Voice Input (Optional)"
            )
            text_input = gr.Textbox(
                label="💬 Text Input",
                placeholder="e.g., List all customers who placed an order in the last 30 days",
                lines=3
            )
            schema_input = gr.Textbox(
                label="📊 Database Schema (Optional)",
                value=default_schema,
                lines=5,
                placeholder="Describe your database tables and columns"
            )
            
            with gr.Row():
                model_selector = gr.Dropdown(
                    choices=list(MODELS.keys()),
                    value="GPT-4.1-mini",
                    label="🤖 Select Model"
                )
                enable_tools = gr.Checkbox(
                    label="🔧 Enable SQL Execution",
                    value=False,
                    info="Allow the AI to execute queries on demo database"
                )
                enable_audio = gr.Checkbox(
                    label="🔊 Audio Response",
                    value=False,
                    info="Get spoken response"
                )
            
            submit_btn = gr.Button("Generate SQL Query", variant="primary")
        
        with gr.Column(scale=3):
            output = gr.Markdown(label="📝 Generated SQL & Explanation")
            audio_output = gr.Audio(label="🔊 Audio Response", autoplay=True)
    
    gr.Examples(
        examples=[
            [None, "List all customers who placed an order in the last 30 days, with their total spend", default_schema, "GPT-4.1-mini", False, False],
            [None, "Show me the top 5 most expensive products", default_schema, "GPT-4.1-mini", False, False],
            [None, "Find customers who haven't placed any orders", default_schema, "Claude Sonnet 4.5", False, False],
            [None, "What is the average order value per customer?", default_schema, "GPT-4.1-mini", True, False],
        ],
        inputs=[audio_input, text_input, schema_input, model_selector, enable_tools, enable_audio]
    )
    
    submit_btn.click(
        fn=process_query,
        inputs=[audio_input, text_input, schema_input, model_selector, enable_tools, enable_audio],
        outputs=[output, audio_output]
    )
    
    text_input.submit(
        fn=process_query,
        inputs=[audio_input, text_input, schema_input, model_selector, enable_tools, enable_audio],
        outputs=[output, audio_output]
    )

demo.launch(inbrowser=True)

In [ ]:
# 替代方案：无需工具的简单流媒体界面

def simple_generate(question, schema, model_name):
    """Simple streaming SQL generation without tool calling"""
    yield from generate_sql_stream(question, schema, model_name)

# 取消下面的注释即可使用更简单的接口，无需调用工具：

# Question_input = gr.Textbox(
# 标签=“您的问题：”，
# placeholder="例如，显示上周的所有订单",
# 行=3
# )
# schema_input = gr.Textbox(
# label="数据库架构：",
# 值=默认模式，
# 行=5
# )
# model_selector = gr.Dropdown(
# 选择=列表(MODELS.keys()),
# 值=“GPT-4.1-mini”，
# 标签=“选择型号”
# )
# 输出 = gr.Markdown(label="生成的 SQL:")

# 视图 = gr. 接口(
# fn=简单生成，
# title="SQL 查询生成器",
# 输入=[问题输入、模式输入、模型选择器]、
# 输出=[输出],
# 例子=[
# [“列出所有客户”，default_schema，“GPT-4.1-mini”]，
# [“显示上个月的订单”，default_schema，“克劳德十四行诗 4.5”]
#     ],
# flagging_mode =“从不”
# )
# view.launch(inbrowser=True)

# 📚 如何使用此应用程序

## 实现的功能：

### ✅ 核心要求：
1. **Gradio UI**：美观、直观的界面，具有多种输入选项
2. **流式传输**：实时响应生成以获得更好的用户体验
3. **系统提示**：专家SQL工程师角色，附有详细说明
4. **型号选择**：GPT-4.1-mini、Claude Sonnet 4.5、Gemini 2.5 Flash、Llama 3.2（本地）之间切换

### ✅ 奖励功能：
5. **工具调用**：启用针对演示数据库的 SQL 执行
6. **音频输入**：使用麦克风说出您的问题
7. **音频输出**：从人工智能获取语音响应

## 使用方法：

### 基本用法：
1. 用自然语言输入您的问题（或使用麦克风）
2. 有选择地修改数据库架构
3. 选择您喜欢的AI模型
4. 单击“生成 SQL 查询”

### 高级功能：
- **启用 SQL 执行**：选中此框可让 AI 在演示数据库上实际运行查询
- **音频响应**：选中此框可听到大声朗读的响应
- **语音输入**：点击麦克风图标通过说话提问

## 示例问题：
- “列出过去 30 天内下过订单的所有客户”
- “平均订单总额是多少？”
- “显示没有订单的客户”
- “找到最昂贵的产品”

## 演示数据库架构：
演示数据库包括：
- **客户**：ID、姓名、电子邮件、创建时间
- **订单**：id、customer_id、总计、order_date  
- **产品**：ID、名称、价格

尝试启用 SQL 执行以查看演示数据库的真实结果！

---

**注意**：确保您在 `.env` 文件中设置了 API 密钥。至少，您需要“OPENAI_API_KEY”。其他键是可选的。